[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/05_Subgraphs_Tests_Loops/Subgraphs_Tests_Loops_Apply.ipynb)

# 1.5 Subgraphs: Tests and Loops — Hands-On Practice

Build and run models with ONNX control flow operators.

---

## Table of Contents

| # | Section | Focus |
|---|---------|-------|
| 1 | [Setup](#section-1) | Imports |
| 2 | [Exercise 1: Custom If Model](#section-2) | Return 100 or -100 based on sum |
| 3 | [Exercise 2: Where Operator](#section-3) | Vectorized element-wise conditional |
| 4 | [Exercise 3: Loop Accumulator](#section-4) | Compute factorial with Loop |
| 5 | [Exercise 4: If vs Where Benchmark](#section-5) | Performance comparison |
| 6 | [Visualization: If Model Flow](#section-6) | Graph structure diagram |
| 7 | [Exercise 5: Nested Conditionals](#section-7) | Multi-class classification |
| 8 | [Challenge: Running Average](#section-8) | Loop-based moving average |

<a id='section-1'></a>
## Section 1: Setup

In [ ]:
# !pip install onnx onnxruntime matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import time

import onnx
from onnx import TensorProto
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid)
from onnx.numpy_helper import from_array
from onnx.checker import check_model
from onnxruntime import InferenceSession

print('Setup complete!')

<a id='section-2'></a>
## Section 2: Exercise 1 — Custom If Model

### Task

Build an `If` model that returns `[100.0]` when $\sum X > 0$, else returns `[-100.0]`.

In [ ]:
zero = from_array(np.array([0], dtype=np.float32), name='zero')
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

rsum = make_node('ReduceSum', ['X'], ['rsum'])
cond = make_node('Greater', ['rsum', 'zero'], ['cond'])

# Then: return [100]
then_out = make_tensor_value_info('then_out', TensorProto.FLOAT, None)
then_node = make_node('Constant', [], ['then_out'],
    value=from_array(np.array([100.0], dtype=np.float32)), name='cst1')
then_body = make_graph([then_node], 'then', [], [then_out])

# Else: return [-100]
else_out = make_tensor_value_info('else_out', TensorProto.FLOAT, [1])
else_node = make_node('Constant', [], ['else_out'],
    value=from_array(np.array([-100.0], dtype=np.float32)), name='cst2')
else_body = make_graph([else_node], 'else', [], [else_out])

if_node = make_node('If', ['cond'], ['Y'],
                    then_branch=then_body, else_branch=else_body)

graph = make_graph([rsum, cond, if_node], 'if_100', [X], [Y], [zero])
model = make_model(graph, opset_imports=[make_opsetid('', 15)])
model.ir_version = 8

sess = InferenceSession(model.SerializeToString(),
                        providers=['CPUExecutionProvider'])

test_cases = [
    ('Positive', np.ones((2, 2), dtype=np.float32)),
    ('Negative', -np.ones((2, 2), dtype=np.float32)),
    ('Mixed (+)', np.array([[5, -1], [2, -3]], dtype=np.float32)),
    ('Mixed (-)', np.array([[-5, 1], [-2, 3]], dtype=np.float32)),
]

print(f'{"Case":>12s} | {"sum(X)":>7s} | {"Result":>8s}')
print('-' * 35)
for name, x_val in test_cases:
    res = sess.run(None, {'X': x_val})[0]
    print(f'{name:>12s} | {x_val.sum():>7.1f} | {res[0]:>8.1f}')

<a id='section-3'></a>
## Section 3: Exercise 2 — Where Operator

### Task

Use the `Where` operator to compute an element-wise sign function: return `1.0` where $X > 0$, else `-1.0`.

In [ ]:
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, None])

ones = from_array(np.array([[1.0]], dtype=np.float32), name='ones')
neg_ones = from_array(np.array([[-1.0]], dtype=np.float32), name='neg_ones')
zero_t = from_array(np.array([[0.0]], dtype=np.float32), name='zero')

graph = make_graph(
    [make_node('Greater', ['X', 'zero'], ['cond']),
     make_node('Where', ['cond', 'ones', 'neg_ones'], ['Y'])],
    'where_model', [X], [Y], [ones, neg_ones, zero_t])
model_w = make_model(graph, opset_imports=[make_opsetid('', 15)])

sess_w = InferenceSession(
    model_w.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.array([[-2, 3, 0], [1, -4, 5]], dtype=np.float32)
result = sess_w.run(None, {'X': x})[0]

print('Where(X > 0, 1, -1):')
print(f'  Input:\n{x}')
print(f'  Output:\n{result}')
print(f'  Correct: {np.allclose(result, np.where(x > 0, 1.0, -1.0))}')

<a id='section-4'></a>
## Section 4: Exercise 3 — Loop Accumulator

### Task

Build a Loop that computes powers: $[X^1, X^2, X^3, \ldots, X^N]$ using repeated multiplication.

In [ ]:
# Body: accum_out = accum_in * X
i_in = make_tensor_value_info('i', TensorProto.INT64, [])
cond_in = make_tensor_value_info('cond_in', TensorProto.BOOL, [])
accum_in = make_tensor_value_info('accum_in', TensorProto.FLOAT, [None])

cond_out = make_tensor_value_info('cond_out', TensorProto.BOOL, [])
accum_out = make_tensor_value_info('accum_out', TensorProto.FLOAT, [None])
scan_out = make_tensor_value_info('scan_out', TensorProto.FLOAT, [None])

body = make_graph(
    [make_node('Mul', ['accum_in', 'X'], ['accum_out']),
     make_node('Identity', ['cond_in'], ['cond_out']),
     make_node('Identity', ['accum_out'], ['scan_out'])],
    'power_body', [i_in, cond_in, accum_in], [cond_out, accum_out, scan_out])

X = make_tensor_value_info('X', TensorProto.FLOAT, [None])
Y_final = make_tensor_value_info('Y_final', TensorProto.FLOAT, [None])
Y_scan = make_tensor_value_info('Y_scan', TensorProto.FLOAT, [None, None])

max_iter = from_array(np.array(6, dtype=np.int64), name='max_iter')
init_cond = from_array(np.array(True), name='init_cond')
init_accum = from_array(np.array([1.0], dtype=np.float32), name='init_accum')

loop_node = make_node('Loop', ['max_iter', 'init_cond', 'init_accum'],
                      ['Y_final', 'Y_scan'], body=body)

graph = make_graph([loop_node], 'power_model', [X], [Y_final, Y_scan],
                   [max_iter, init_cond, init_accum])
model_pow = make_model(graph, opset_imports=[make_opsetid('', 15)])
model_pow.ir_version = 8

sess_pow = InferenceSession(
    model_pow.SerializeToString(), providers=['CPUExecutionProvider'])

for base in [2.0, 3.0, 0.5]:
    x_val = np.array([base], dtype=np.float32)
    y_final, y_scan = sess_pow.run(None, {'X': x_val})
    powers = y_scan.flatten()
    expected = [base**i for i in range(1, 7)]
    print(f'  X={base}: powers = {powers}')
    print(f'        expected = {expected}')
    print(f'        match: {np.allclose(powers, expected)}')

<a id='section-5'></a>
## Section 5: Exercise 4 — If vs Where Performance

### Task

Compare the performance of the If-based model vs the Where-based model for classifying tensor elements.

In [ ]:
# Build a Where model for benchmarking
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, None])

pos_val = from_array(np.array([[1.0]], dtype=np.float32), name='pos')
neg_val = from_array(np.array([[-1.0]], dtype=np.float32), name='neg')
zero_ref = from_array(np.array([[0.0]], dtype=np.float32), name='zero')

graph_where = make_graph(
    [make_node('Greater', ['X', 'zero'], ['c']),
     make_node('Where', ['c', 'pos', 'neg'], ['Y'])],
    'bench_where', [X], [Y], [pos_val, neg_val, zero_ref])
model_bench = make_model(graph_where, opset_imports=[make_opsetid('', 15)])

sess_bench = InferenceSession(
    model_bench.SerializeToString(), providers=['CPUExecutionProvider'])

sizes = [10, 100, 1000, 10000]
where_times = []
numpy_times = []

for n in sizes:
    x_data = np.random.randn(n, n).astype(np.float32)

    # Warm up
    sess_bench.run(None, {'X': x_data})
    np.where(x_data > 0, 1.0, -1.0)

    runs = 50
    t0 = time.perf_counter()
    for _ in range(runs):
        sess_bench.run(None, {'X': x_data})
    where_times.append((time.perf_counter() - t0) / runs * 1000)

    t0 = time.perf_counter()
    for _ in range(runs):
        np.where(x_data > 0, 1.0, -1.0)
    numpy_times.append((time.perf_counter() - t0) / runs * 1000)

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(sizes))
w = 0.35
ax.bar(x_pos - w/2, where_times, w, label='ONNX Where', color='#3498DB')
ax.bar(x_pos + w/2, numpy_times, w, label='NumPy where', color='#E74C3C')
ax.set_xlabel('Matrix Size (N×N)', fontsize=12)
ax.set_ylabel('Time per call (ms)', fontsize=12)
ax.set_title('Where Operator: ONNX Runtime vs NumPy', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{n}×{n}' for n in sizes])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

<a id='section-6'></a>
## Section 6: Visualization — If Model Structure

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('off')
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 10)
ax.set_title('If Model: Conditional Branching Architecture', fontsize=14, fontweight='bold')

s = {
    'in': dict(boxstyle='round,pad=0.5', fc='#AED6F1', ec='#2471A3', lw=2),
    'op': dict(boxstyle='round,pad=0.5', fc='#F9E79F', ec='#B7950B', lw=2),
    'br': dict(boxstyle='round,pad=0.5', fc='#FADBD8', ec='#C0392B', lw=2),
    'out': dict(boxstyle='round,pad=0.5', fc='#A9DFBF', ec='#1E8449', lw=2),
}
arr = dict(arrowstyle='->', lw=2, color='#555')

ax.text(5, 9, 'X\n(N×D float)', ha='center', fontsize=10, fontweight='bold', bbox=s['in'])
ax.text(5, 7, 'ReduceSum', ha='center', fontsize=10, fontweight='bold', bbox=s['op'])
ax.text(5, 5.5, 'Greater(·, 0)\n→ cond (bool)', ha='center', fontsize=9, bbox=s['op'])
ax.text(2, 3.5, 'then_branch\nConstant([100])', ha='center', fontsize=9, bbox=s['br'])
ax.text(8, 3.5, 'else_branch\nConstant([-100])', ha='center', fontsize=9, bbox=s['br'])
ax.text(5, 1.5, 'Y\n(result)', ha='center', fontsize=10, fontweight='bold', bbox=s['out'])

ax.annotate('', xy=(5, 7.8), xytext=(5, 8.3), arrowprops=arr)
ax.annotate('', xy=(5, 6.3), xytext=(5, 6.5), arrowprops=arr)
ax.annotate('', xy=(2, 4.3), xytext=(4, 5), arrowprops=dict(**arr, color='green'))
ax.annotate('', xy=(8, 4.3), xytext=(6, 5), arrowprops=dict(**arr, color='red'))
ax.annotate('', xy=(5, 2.2), xytext=(2, 2.8), arrowprops=arr)
ax.annotate('', xy=(5, 2.2), xytext=(8, 2.8), arrowprops=arr)

ax.text(3, 5.3, 'True', fontsize=10, color='green', fontweight='bold')
ax.text(6.5, 5.3, 'False', fontsize=10, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

<a id='section-7'></a>
## Section 7: Exercise 5 — Clamp with Where

### Task

Use nested `Where` operators to implement a clamp function: $\text{clamp}(X, \text{lo}, \text{hi}) = \min(\max(X, \text{lo}), \text{hi})$.

In [ ]:
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, None])

lo = from_array(np.array([[-2.0]], dtype=np.float32), name='lo')
hi = from_array(np.array([[2.0]], dtype=np.float32), name='hi')

graph = make_graph(
    [make_node('Less', ['X', 'lo'], ['below']),
     make_node('Where', ['below', 'lo', 'X'], ['clipped_lo']),
     make_node('Greater', ['clipped_lo', 'hi'], ['above']),
     make_node('Where', ['above', 'hi', 'clipped_lo'], ['Y'])],
    'clamp', [X], [Y], [lo, hi])
model_clamp = make_model(graph, opset_imports=[make_opsetid('', 15)])

sess_clamp = InferenceSession(
    model_clamp.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.array([[-5, -2, -1, 0, 1, 2, 5]], dtype=np.float32)
result = sess_clamp.run(None, {'X': x})[0]

print('Clamp(X, lo=-2, hi=2):')
print(f'  Input:    {x[0]}')
print(f'  Clamped:  {result[0]}')
print(f'  Expected: {np.clip(x, -2, 2)[0]}')
print(f'  Match: {np.allclose(result, np.clip(x, -2, 2))}')

<a id='section-8'></a>
## Section 8: Challenge — Running Average with Loop

### Task

Build a Loop-based model that computes a running average: at iteration $t$, the output is $\frac{1}{t+1}\sum_{i=0}^{t} X$.

In [ ]:
# Body: sum_out = sum_in + X, scan_out = sum_out / (i + 1)
i_in = make_tensor_value_info('i', TensorProto.INT64, [])
cond_in = make_tensor_value_info('cond_in', TensorProto.BOOL, [])
sum_in = make_tensor_value_info('sum_in', TensorProto.FLOAT, [None])

cond_out = make_tensor_value_info('cond_out', TensorProto.BOOL, [])
sum_out = make_tensor_value_info('sum_out', TensorProto.FLOAT, [None])
avg_out = make_tensor_value_info('avg_out', TensorProto.FLOAT, [None])

one_i = from_array(np.array(1, dtype=np.int64), name='one_i')

body = make_graph(
    [make_node('Add', ['sum_in', 'X'], ['sum_out']),
     make_node('Identity', ['cond_in'], ['cond_out']),
     make_node('Add', ['i', 'one_i'], ['ip1']),
     make_node('Cast', ['ip1'], ['ip1_f'], to=1),
     make_node('Div', ['sum_out', 'ip1_f'], ['avg_out'])],
    'avg_body',
    [i_in, cond_in, sum_in],
    [cond_out, sum_out, avg_out],
    [one_i])

X = make_tensor_value_info('X', TensorProto.FLOAT, [None])
Y_sum = make_tensor_value_info('Y_sum', TensorProto.FLOAT, [None])
Y_avgs = make_tensor_value_info('Y_avgs', TensorProto.FLOAT, [None, None])

n_steps = 10
max_iter = from_array(np.array(n_steps, dtype=np.int64), name='max_iter')
init_cond = from_array(np.array(True), name='init_cond')
init_sum = from_array(np.array([0.0], dtype=np.float32), name='init_sum')

loop_node = make_node('Loop', ['max_iter', 'init_cond', 'init_sum'],
                      ['Y_sum', 'Y_avgs'], body=body)

graph = make_graph([loop_node], 'running_avg', [X], [Y_sum, Y_avgs],
                   [max_iter, init_cond, init_sum])
model_avg = make_model(graph, opset_imports=[make_opsetid('', 15)])
model_avg.ir_version = 8

sess_avg = InferenceSession(
    model_avg.SerializeToString(), providers=['CPUExecutionProvider'])

x_val = np.array([3.0], dtype=np.float32)
y_sum, y_avgs = sess_avg.run(None, {'X': x_val})
avgs = y_avgs.flatten()

print(f'Running average of X={x_val[0]} over {n_steps} iterations:')
print(f'  Averages: {avgs}')
print(f'  Expected: all {x_val[0]} (since same X added each time)')
print(f'  Final sum: {y_sum[0]}')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, n_steps + 1), avgs, 'o-', color='#3498DB', markersize=8, linewidth=2)
ax.axhline(y=x_val[0], color='red', linestyle='--', label=f'True average = {x_val[0]}')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Running Average', fontsize=12)
ax.set_title('Loop-Based Running Average', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Summary

| Exercise | Skill |
|----------|-------|
| 1 | Build If model with custom return values |
| 2 | Element-wise conditional with Where |
| 3 | Loop-based power computation |
| 4 | Performance benchmarking |
| 5 | Nested Where for clamping |
| Challenge | Running average with Loop |

**Next:** [Functions](../06_Functions/) — Define reusable operator combinations.